In [ ]:
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr

import scipy
from scipy.stats import t

import pandas as pd
import xskillscore as xs
import numpy as np
import dask
from dask.distributed import Client

import albedo_functions as af

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
COV_PATH = f'{POST_DATA}/'
LAI_PATH = f'{CONFESS_DATA}/'
SAVE_PATH = f'{WORK_DIR}/'

In [ ]:
# Open datasets with Dask chunking
cvh_sens = xr.open_dataset(COV_PATH + f'{exp_sens}_effective_cvh_199311-201910_1x1.nc', chunks=-1)['cvh']
cvl_sens = xr.open_dataset(COV_PATH + f'{exp_sens}_effective_cvl_199311-201910_1x1.nc', chunks=-1)['cvl']
lai_hv_a52o = xr.open_dataset(LAI_PATH + f'{exp_sens}/original_resolution/lai_hv/{exp_sens}_lai_hv_199311-201910_1x1.nc', chunks=-1)['LAI_HV']
lai_lv_a52o = xr.open_dataset(LAI_PATH + f'{exp_sens}/original_resolution/lai_lv/{exp_sens}_lai_lv_199311-201910_1x1.nc', chunks=-1)['LAI_LV']


cvh_ctrl = xr.open_dataset(COV_PATH + f'{exp_ctrl}_effective_cvh_1x1.nc', chunks=-1)['cvh']
cvl_ctrl = xr.open_dataset(COV_PATH + f'{exp_ctrl}_effective_cvl_1x1.nc', chunks=-1)['cvl']
a1ua_land_file = f'{BC_DATA}/'+exp_ctrl+'/icmcl_a1ua_regular_1x1.nc'
dset_a1ua = xr.open_dataset(a1ua_land_file, chunks=-1)
lai_hv_a1ua = dset_a1ua['lai_hv'].sel(time=slice(lai_hv_a52o.time[0],lai_hv_a52o.time[-1]))
lai_lv_a1ua = dset_a1ua['lai_lv'].sel(time=slice(lai_hv_a52o.time[0],lai_hv_a52o.time[-1]))

effective_laih_ctrl = lai_hv_a1ua * cvh_ctrl
effective_lail_ctrl = lai_lv_a1ua * cvl_ctrl
effective_laih_sens = lai_hv_a52o * cvh_sens
effective_lail_sens = lai_lv_a52o * cvl_sens

effective_laih_ctrl['time'] = pd.to_datetime(effective_laih_ctrl['time'].values).normalize().to_period('M').start_time
effective_laih_sens['time'] = pd.to_datetime(effective_laih_sens['time'].values).normalize().to_period('M').start_time
effective_lail_ctrl['time'] = pd.to_datetime(effective_lail_ctrl['time'].values).normalize().to_period('M').start_time
effective_lail_sens['time'] = pd.to_datetime(effective_lail_sens['time'].values).normalize().to_period('M').start_time

#select time from 1999
start_date = '1999-09-01'

laih_ctrl = lai_hv_a1ua.sel(time=slice(start_date, None))
laih_sens = lai_hv_a52o.sel(time=slice(start_date, None))
lail_ctrl = lai_lv_a1ua.sel(time=slice(start_date, None))
lail_sens = lai_lv_a52o.sel(time=slice(start_date, None))

cvh_ctrl = cvh_ctrl.sel(time=slice(start_date, None))
cvh_sens = cvh_sens.sel(time=slice(start_date, None))
cvl_ctrl = cvl_ctrl.sel(time=slice(start_date, None))
cvl_sens = cvl_sens.sel(time=slice(start_date, None))

effective_laih_ctrl = lai_hv_a1ua.sel(time=slice(start_date, None))
effective_laih_sens = lai_hv_a52o.sel(time=slice(start_date, None))
effective_lail_ctrl = lai_lv_a1ua.sel(time=slice(start_date, None))
effective_lail_sens = lai_lv_a52o.sel(time=slice(start_date, None))


In [ ]:
# Convert time to numeric (e.g., years)
time_sens = xr.DataArray(
    np.arange(laih_sens.sizes["time"]),
    dims="time"
)

time_ctrl = xr.DataArray(
    np.arange(laih_ctrl.sizes["time"]),
    dims="time"
)

In [ ]:
from scipy import stats
def trend(ds, time):
    slope, intercept, r_value, p_value, std_err = stats.linregress(time, ds)
    return slope, p_value, std_err

In [ ]:
laih_ctrl_slope, laih_ctrl_p_value, laih_ctrl_std_err = xr.apply_ufunc(
    trend,
    laih_ctrl.chunk(dict(time=-1)),
    time_ctrl,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[], [], []],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float, float, float],
)

laih_sens_slope, laih_sens_p_value, laih_sens_std_err = xr.apply_ufunc(
    trend,
    laih_sens.chunk(dict(time=-1)),
    time_sens,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[],[], []],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float, float, float],
)

laih_ctrl_slope.to_dataset(name='slope').to_netcdf(f'{SAVE_PATH}a1ua_laih_trend_1999.nc')
laih_sens_slope.to_dataset(name='slope').to_netcdf(f'{SAVE_PATH}a52o_laih_trend_1999.nc')
laih_ctrl_p_value.to_dataset(name='p').to_netcdf(f'{SAVE_PATH}a1ua_laih_trend_p_1999.nc')
laih_sens_p_value.to_dataset(name='p').to_netcdf(f'{SAVE_PATH}a52o_laih_trend_p_1999.nc')

In [ ]:
lail_ctrl_slope, lail_ctrl_p_value, lail_ctrl_std_err = xr.apply_ufunc(
    trend,
    lail_ctrl.chunk(dict(time=-1)),
    time_ctrl,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[], [], []],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float, float, float],
)

lail_sens_slope, lail_sens_p_value, lail_sens_std_err = xr.apply_ufunc(
    trend,
    lail_sens.chunk(dict(time=-1)),
    time_sens,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[],[], []],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float, float, float],
)

lail_ctrl_slope.to_dataset(name='slope').to_netcdf(f'{SAVE_PATH}a1ua_lail_trend_1999.nc')
lail_sens_slope.to_dataset(name='slope').to_netcdf(f'{SAVE_PATH}a52o_lail_trend_1999.nc')
lail_ctrl_p_value.to_dataset(name='p').to_netcdf(f'{SAVE_PATH}a1ua_lail_trend_p_1999.nc')
lail_sens_p_value.to_dataset(name='p').to_netcdf(f'{SAVE_PATH}a52o_lail_trend_p_1999.nc')

In [ ]:
# Convert time to numeric (e.g., years)
time_sens = xr.DataArray(
    np.arange(cvh_sens.sizes["time"]),
    dims="time"
)

time_ctrl = xr.DataArray(
    np.arange(cvh_ctrl.sizes["time"]),
    dims="time"
)

In [ ]:
cvh_ctrl_slope, cvh_ctrl_p_value, cvh_ctrl_std_err = xr.apply_ufunc(
    trend,
    cvh_ctrl.chunk(dict(time=-1)),
    time_ctrl,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[], [], []],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float, float, float],
)

cvh_sens_slope, cvh_sens_p_value, cvh_sens_std_err = xr.apply_ufunc(
    trend,
    cvh_sens.chunk(dict(time=-1)),
    time_sens,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[],[], []],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float, float, float],
)

cvh_ctrl_slope.to_dataset(name='slope').to_netcdf(f'{SAVE_PATH}a1ua_cvh_trend_1999.nc')
cvh_sens_slope.to_dataset(name='slope').to_netcdf(f'{SAVE_PATH}a52o_cvh_trend_1999.nc')
cvh_ctrl_p_value.to_dataset(name='p').to_netcdf(f'{SAVE_PATH}a1ua_cvh_trend_p_1999.nc')
cvh_sens_p_value.to_dataset(name='p').to_netcdf(f'{SAVE_PATH}a52o_cvh_trend_p_1999.nc')

In [ ]:
cvl_ctrl_slope, cvl_ctrl_p_value, cvl_ctrl_std_err = xr.apply_ufunc(
    trend,
    cvl_ctrl.chunk(dict(time=-1)),
    time_ctrl,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[], [], []],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float, float, float],
)

cvl_sens_slope, cvl_sens_p_value, cvl_sens_std_err = xr.apply_ufunc(
    trend,
    cvl_sens.chunk(dict(time=-1)),
    time_sens,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[],[], []],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float, float, float],
)

cvl_ctrl_slope.to_dataset(name='slope').to_netcdf(f'{SAVE_PATH}a1ua_cvl_trend_1999.nc')
cvl_sens_slope.to_dataset(name='slope').to_netcdf(f'{SAVE_PATH}a52o_cvl_trend_1999.nc')
cvl_ctrl_p_value.to_dataset(name='p').to_netcdf(f'{SAVE_PATH}a1ua_cvl_trend_p_1999.nc')
cvl_sens_p_value.to_dataset(name='p').to_netcdf(f'{SAVE_PATH}a52o_cvl_trend_p_1999.nc')

In [ ]:
# Convert time to numeric (e.g., years)
time_sens = xr.DataArray(
    np.arange(effective_laih_sens.sizes["time"]),
    dims="time"
)

time_ctrl = xr.DataArray(
    np.arange(effective_laih_ctrl.sizes["time"]),
    dims="time"
)

In [ ]:
effective_laih_ctrl_slope, effective_laih_ctrl_p_value, effective_laih_ctrl_std_err = xr.apply_ufunc(
    trend,
    effective_laih_ctrl.chunk(dict(time=-1)),
    time_ctrl,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[], [], []],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float, float, float],
)

effective_laih_sens_slope, effective_laih_sens_p_value, effective_laih_sens_std_err = xr.apply_ufunc(
    trend,
    effective_laih_sens.chunk(dict(time=-1)),
    time_sens,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[],[], []],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float, float, float],
)

effective_laih_ctrl_slope.to_dataset(name='slope').to_netcdf(f'{SAVE_PATH}a1ua_effective_laih_trend_1999.nc')
effective_laih_sens_slope.to_dataset(name='slope').to_netcdf(f'{SAVE_PATH}a52o_effective_laih_trend_1999.nc')
effective_laih_ctrl_p_value.to_dataset(name='p').to_netcdf(f'{SAVE_PATH}a1ua_effective_laih_trend_p_1999.nc')
effective_laih_sens_p_value.to_dataset(name='p').to_netcdf(f'{SAVE_PATH}a52o_effective_laih_trend_p_1999.nc')

In [ ]:
effective_lail_ctrl_slope, effective_lail_ctrl_p_value, effective_lail_ctrl_std_err = xr.apply_ufunc(
    trend,
    effective_lail_ctrl.chunk(dict(time=-1)),
    time_ctrl,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[], [], []],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float, float, float],
)

effective_lail_sens_slope, effective_lail_sens_p_value, effective_lail_sens_std_err = xr.apply_ufunc(
    trend,
    effective_lail_sens.chunk(dict(time=-1)),
    time_sens,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[[],[], []],
    vectorize=True,
    dask="parallelized",
    output_dtypes=[float, float, float],
)

effective_lail_ctrl_slope.to_dataset(name='slope').to_netcdf(f'{SAVE_PATH}a1ua_effective_lail_trend_1999.nc')
effective_lail_sens_slope.to_dataset(name='slope').to_netcdf(f'{SAVE_PATH}a52o_effective_lail_trend_1999.nc')
effective_lail_ctrl_p_value.to_dataset(name='p').to_netcdf(f'{SAVE_PATH}a1ua_effective_lail_trend_p_1999.nc')
effective_lail_sens_p_value.to_dataset(name='p').to_netcdf(f'{SAVE_PATH}a52o_effective_lail_trend_p_1999.nc')

In [ ]:
def delta_sign(n1, slope1, se1, n2, slope2, se2):
    """
    Compare two regression slopes.

    Parameters:
        n1, n2 (int): Number of observations in each dataset
        slope1, slope2 (float): Slopes from the two regressions
        se1, se2 (float): Standard errors of the slopes

    Returns:
        float: Two-tailed p-value for the difference in slopes
    """
    df = n1 + n2 - 4
    t_stat = (slope1 - slope2) / (se1**2 + se2**2)**0.5
    p_value = 2 * (1 - t.cdf(abs(t_stat), df=df))
    return p_value

In [ ]:
p_delta = delta_sign(len(laih_ctrl.time), laih_ctrl_slope, laih_ctrl_std_err, len(laih_sens.time), laih_sens_slope, laih_sens_std_err)

p_delta = xr.DataArray(p_delta, 
                       name='p_delta',
                       dims=['lat', 'lon'],
                       coords={'lat': laih_sens_slope.lat, 'lon': laih_sens_slope.lon})

p_delta.to_netcdf(f'{SAVE_PATH}delta_laih_trend_p_1999.nc')

In [ ]:
p_delta = delta_sign(len(lail_ctrl.time), lail_ctrl_slope, lail_ctrl_std_err, len(lail_sens.time), lail_sens_slope, lail_sens_std_err)

p_delta = xr.DataArray(p_delta, 
                       name='p_delta',
                       dims=['lat', 'lon'],
                       coords={'lat': lail_sens_slope.lat, 'lon': lail_sens_slope.lon})

p_delta.to_netcdf(f'{SAVE_PATH}delta_lail_trend_p_1999.nc')

In [ ]:
p_delta = delta_sign(len(cvh_ctrl.time), cvh_ctrl_slope, cvh_ctrl_std_err, len(cvh_sens.time), cvh_sens_slope, cvh_sens_std_err)

p_delta = xr.DataArray(p_delta, 
                       name='p_delta',
                       dims=['lat', 'lon'],
                       coords={'lat': cvh_sens_slope.lat, 'lon': cvh_sens_slope.lon})

p_delta.to_netcdf(f'{SAVE_PATH}delta_cvh_trend_p_1999.nc')

In [ ]:
p_delta = delta_sign(len(cvl_ctrl.time), cvl_ctrl_slope, cvl_ctrl_std_err, len(cvl_sens.time), cvl_sens_slope, cvl_sens_std_err)

p_delta = xr.DataArray(p_delta, 
                       name='p_delta',
                       dims=['lat', 'lon'],
                       coords={'lat': cvl_sens_slope.lat, 'lon': cvl_sens_slope.lon})

p_delta.to_netcdf(f'{SAVE_PATH}delta_cvl_trend_p_1999.nc')

In [ ]:
p_delta = delta_sign(len(effective_laih_ctrl.time), effective_laih_ctrl_slope, effective_laih_ctrl_std_err, len(effective_laih_sens.time), effective_laih_sens_slope, effective_laih_sens_std_err)

p_delta = xr.DataArray(p_delta, 
                       name='p_delta',
                       dims=['lat', 'lon'],
                       coords={'lat': effective_laih_sens_slope.lat, 'lon': effective_laih_sens_slope.lon})

p_delta.to_netcdf(f'{SAVE_PATH}delta_effective_laih_trend_p_1999.nc')

In [ ]:
p_delta = delta_sign(len(effective_lail_ctrl.time), effective_lail_ctrl_slope, effective_lail_ctrl_std_err, len(effective_lail_sens.time), effective_lail_sens_slope, effective_lail_sens_std_err)

p_delta = xr.DataArray(p_delta, 
                       name='p_delta',
                       dims=['lat', 'lon'],
                       coords={'lat': effective_lail_sens_slope.lat, 'lon': effective_lail_sens_slope.lon})

p_delta.to_netcdf(f'{SAVE_PATH}delta_effective_lail_trend_p_1999.nc')